# ⚗  모델 성능 평가 및 비교
이 노트북은 `llama3.1`, `qwen2.5`, `gemma4` 등 여러 Ollama 모델의 프롬프트 응답 품질과 처리 속도를 비교하기 위해 원본 파이프라인에서 분리된 파일입니다.

In [1]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# 2. Ollama 설치 및 백그라운드 실행, 평가용 모델 다운로드
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# 서버 실행
process = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

!pip install ollama pandas

# 평가할 모델들 다운로드
!ollama pull llama3.1
!ollama pull qwen2.5
!ollama pull gemma4


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Package

In [5]:
# 3. 모델별 실제 분석 결과 상세 비교
import json
import re
import ollama
import pandas as pd
from IPython.display import display

# 실제 데이터 불러오기
input_path = "/content/drive/MyDrive/MovieReview/data/cleaned_total_reviews.json"
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

real_reviews = []
for m_name, m_data in data.items():
    real_reviews.extend(m_data.get("reviews", []))

# 더미 데이터 제외
real_reviews = [r for r in real_reviews if "mock review" not in r.get("content", "").lower()]

# 샘플링
sample_reviews = real_reviews[:3]
models = ["llama3.1", "qwen2.5", "gemma4"]

print("🔍 모델별 실제 분석 결과 상세 비교 (샘플 3개)\n" + "="*60)

# 프롬프트용 본문 텍스트 생성
batch_text = ""
for idx, r in enumerate(sample_reviews):
    batch_text += f"ID:{idx} | 본문: {r['content']}\n"

prompt = f""""
당신은 영화 리뷰 분석 전문가입니다. 다음 {len(sample_reviews)}개의 리뷰를 각각 분석하여 정보를 추출하세요.
응답은 반드시 한국어로 작성하세요.

[범주]: 전체긍정, 전체부정, 전체복합, 연기좋음, 연기나쁨, 연출좋음, 연출나쁨, 서사좋음, 서사나쁨, 비주얼좋음, 비주얼나쁨, 음악좋음, 분위기가벼움, 분위기무거움, 고증좋음, 고증나쁨, 주의사항, TMI, 장르특성

[분석 지침]:
- 각 리뷰의 본문을 꼼꼼히 읽고 해당되는 모든 범주를 선택하세요.
- 가장 먼저 리뷰의 전체적인 총평이나 분위기가 긍정적이면 '전체긍정', 부정적이면 '전체부정', 장단점이 섞여있거나 모호하면 '전체복합' 중 하나를 반드시 포함하세요.
- ⚠️ 절대 비워두지 마세요: 모든 리뷰에 대해 반드시 최소 1개 이상의 'content_character'를 찾아야 합니다.
- ⚠️ 절대 비워두지 마세요: 'search_keywords'는 본문 내용에서 유추할 수 있는 대표 핵심 단어를 최대한 다양하고 풍부하게 추출하세요.

[응답 지시 (매우 중요)]:
1. 반드시 아래 제시된 JSON 리스트 형식으로만 응답해야 합니다. 다른 부가적인 텍스트는 절대 포함하지 마세요.
2. JSON 문법을 완벽하게 지켜야 합니다.

형식 예시:
[
    {{"id": 0, "content_character": ["전체긍정", "연출좋음"], "search_keywords": ["핵심단어1", "핵심단어2"]}}
]

[리뷰 리스트]:
{batch_text}
"""

# 결과를 저장할 딕셔너리
results_dict = {i: {"리뷰 본문": sample_reviews[i]["content"]} for i in range(len(sample_reviews))}

# 각 모델별로 추론 실행
for model in models:
    print(f"▶ {model} 모델 추론 중...")
    try:
        response = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
        json_text = re.sub(r'```json|```', '', response['message']['content']).strip()
        parsed = json.loads(json_text)

        for item in parsed:
            rid = item.get("id")
            if rid in results_dict:
                results_dict[rid][f"{model}\n태그"] = ", ".join(item.get("content_character", []))
                results_dict[rid][f"{model}\n키워드"] = ", ".join(item.get("search_keywords", []))
    except Exception as e:
        print(f"⚠️ {model} 오류 발생: {e}")

print("="*60 + "\n✨ 분석 완료! 결과 데이터프레임:")

df = pd.DataFrame(list(results_dict.values()))
df = df.fillna("N/A")
display(df)


🔍 모델별 실제 분석 결과 상세 비교 (샘플 3개)
▶ llama3.1 모델 추론 중...
▶ qwen2.5 모델 추론 중...
⚠️ qwen2.5 오류 발생: Expecting ',' delimiter: line 2 column 74 (char 75)
▶ gemma4 모델 추론 중...
⚠️ gemma4 오류 발생: llama runner process has terminated: %!w(<nil>) (status code: 500)
✨ 분석 완료! 결과 데이터프레임:


,리뷰 본문,llama3.1\n태그,llama3.1\n키워드
0,"극장만이 선사하는 가치, 블록버스터의 존재 이유.","전체긍정, 연출좋음","블록버스터, 극장"
1,영화관의 위기를 우려하는 이들에게 분명하게 증명한다. 극장에서만 체감할 수 있는 쾌...,"전체긍정, 서사좋음","극장, 쾌감"
2,보이는 그대로 얻는 데서 오는 포만감.,"전체긍정, 비주얼좋음","보이는, 포만감"
